# 0. Arithmetic

No derivatives and no integrals here: only `+`, `-`, `*`, `/` and powers, on
ordinary numbers. Two questions:

1. **Does ordinary arithmetic still work?** Yes, and the value you read back is the
   same float you would have computed.
2. **What do you get on top of it?** Everything that ordinary arithmetic throws away
   at zero: what was multiplied by zero, what cancelled, how fast something vanished
   or blew up, and the ratio between two zeros.

Each extra is shown on a small task, then what it costs, and how to switch it off.

In [1]:
import time, warnings
import composite.composite_lib as cl
from composite import R, ZERO, INF, Composite
from composite.composite_lib import conventional
from composite.backends.config import use_dict

use_dict(); cl._refresh_constants()
warnings.simplefilter("ignore")          # the library reports every conversion; see section 9
print("backend: dict")

backend: dict


## 1. Ordinary numbers stay ordinary

`R(x)` makes a composite from an ordinary number. Arithmetic on composites that never
meet a zero gives exactly the float result, held at grade 0 (`|v|_0`). Python numbers
mix in freely.

In [2]:
cases = [("3 + 4",     R(3) + R(4),     3 + 4),
         ("3 * 4",     R(3) * R(4),     3 * 4),
         ("3 / 4",     R(3) / R(4),     3 / 4),
         ("2 ** 10",   R(2) ** 10,      2 ** 10),
         ("7 - 2.5",   R(7) - 2.5,      7 - 2.5),
         ("10 / R(4)", 10 / R(4),       10 / 4),
         ("0.1 + 0.2", R(0.1) + R(0.2), 0.1 + 0.2)]
print("%-12s %-22s %-22s %s" % ("expression", "composite", "float", "same value"))
for label, c, f in cases:
    print("%-12s %-22s %-22r %s" % (label, c, f, c.st() == f))

expression   composite              float                  same value
3 + 4        <|7|₀>                 7                      True
3 * 4        <|12|₀>                12                     True
3 / 4        <|0.75|₀>              0.75                   True
2 ** 10      <|1024|₀>              1024                   True
7 - 2.5      <|4.5|₀>               4.5                    True
10 / R(4)    <|2.5|₀>               2.5                    True
0.1 + 0.2    <|0.3|₀>               0.30000000000000004    True


The last row matters. The composite prints `0.3` only because the display rounds to
six digits; its value is `0.30000000000000004`, the same as the float. Composites store
float64 coefficients, so they do **not** fix floating-point rounding. What they add is
somewhere else.

## 2. Reading a result back

* `st()` is the standard part: the ordinary value, grade 0.
* `to_ieee754()` is the image in float arithmetic: the value if there is one, `0.0`
  for something infinitesimal, `+-inf` for something infinite, `nan` for nothing.
* `float()` is `st()`, and refuses a number that has no standard part, which is what
  catches a composite falling into `math.*` by accident. For NOTHING it returns `0.0`;
  `to_ieee754()` is the one that says `nan`, because NOTHING is not a value.

In [3]:
for label, c in (("R(2.5)", R(2.5)), ("6 - 6", R(6) - R(6)), ("1 / 0", R(1) / R(0)),
                 ("NOTHING", Composite({}))):
    try:
        f = float(c)
    except Exception as e:
        f = type(e).__name__
    print("%-8s %-10s to_ieee754 %-6s float() %s" % (label, c, c.to_ieee754(), f))

R(2.5)   <|2.5|₀>   to_ieee754 2.5    float() 2.5
6 - 6    <|6|₋₁>    to_ieee754 0.0    float() 0.0
1 / 0    <|1|₁>     to_ieee754 inf    float() StandardPartUndefinedError
NOTHING  ∅          to_ieee754 nan    float() 0.0


## 3. Multiplying by zero is reversible

Conventionally `5 * 0 = 0` and the 5 is gone: `5 * 0 / 0` is undefined. Here `* 0`
moves the number down one grade, and `/ 0` moves it back.

In [4]:
x = R(5)
steps = [("5", x), ("5 * 0", x * 0), ("5 * 0 * 0", x * 0 * 0),
         ("5 * 0 * 0 / 0", x * 0 * 0 / 0), ("5 * 0 * 0 / 0 / 0", x * 0 * 0 / 0 / 0)]
for label, v in steps:
    print("%-20s %s" % (label, v))

5                    <|5|₀>
5 * 0                <|5|₋₁>
5 * 0 * 0            <|5|₋₂>
5 * 0 * 0 / 0        <|5|₋₁>
5 * 0 * 0 / 0 / 0    <|5|₀>


## 4. Division by zero continues

A zero divisor gives an infinity **of a definite order and size** instead of an
error. The grade is the order: `|5|_1` is five first-order infinities, `|1|_2` is a
second-order one. Infinities add, subtract and divide like any other number.

In [5]:
print("1 / 0            ", R(1) / R(0))
print("5 / 0            ", R(5) / R(0))
print("1 / (0 * 0)      ", R(1) / (R(0) * R(0)))
print()
print("a rational function at its pole: (x + 1) / (x - 1) at x = 1")
x = R(1)
print("  conventional   ", "ZeroDivisionError")
print("  composite      ", (x + 1) / (x - R(1)), "  a first-order pole, residue 2")
print()
print("3*INF - INF      ", 3 * INF - INF, "  infinities subtract and keep their order")
print("INF / INF        ", INF / INF)
print("INF * 0          ", INF * R(0))

1 / 0             <|1|₁>
5 / 0             <|5|₁>
1 / (0 * 0)       <|1|₂>

a rational function at its pole: (x + 1) / (x - 1) at x = 1
  conventional    ZeroDivisionError
  composite       <|2|₁>   a first-order pole, residue 2

3*INF - INF       <|2|₁>   infinities subtract and keep their order
INF / INF         <|1|₀>
INF * 0           <|1|₀>


## 5. Zeros keep their ratios

Euler's point (*Institutiones calculi differentialis*, 1755, section 85): any two
zeros are equal arithmetically, but their ratio can be anything, and losing it is
what makes `0/0` indeterminate. A cancellation here keeps the quantity it
annihilated, so the ratio survives.

In [6]:
print("(2 - 2) / (3 - 3) =", (R(2) - R(2)) / (R(3) - R(3)), "   2/3")
print("(6 - 6) / (2 - 2) =", (R(6) - R(6)) / (R(2) - R(2)), "   3")
print("0 / 0             =", R(0) / R(0), "   two written zeros of the same order")
print("0 / (0 * 0)       =", R(0) / (R(0) * R(0)), "   a slower zero over a faster one")

(2 - 2) / (3 - 3) = <|0.666667|₀>    2/3
(6 - 6) / (2 - 2) = <|3|₀>    3
0 / 0             = <|1|₀>    two written zeros of the same order
0 / (0 * 0)       = <|1|₁>    a slower zero over a faster one


### A practical case: a weighted average whose weights are all zero

`sum(w*x) / sum(w)` with every weight zero is `0/0`: conventionally `nan`, and code
has to special-case it. Here each `w * x` keeps `x` one grade down, the denominator
keeps a count of the zeros, and the ratio is the **unweighted** mean. Nothing in the
code above the division knows the weights were zero.

In [7]:
def weighted_mean(xs, ws):
    num, den = Composite({}), Composite({})       # start from NOTHING; see section 7
    for x, w in zip(xs, ws):
        num = num + R(w) * R(x)
        den = den + R(w)
    return num / den

xs = [2.0, 4.0, 9.0]
print("weights 0, 0, 0 ->", weighted_mean(xs, [0.0, 0.0, 0.0]), "  plain mean", sum(xs) / 3)
print("weights 1, 1, 1 ->", weighted_mean(xs, [1.0, 1.0, 1.0]))
m = weighted_mean(xs, [1.0, 0.0, 1.0])
print("weights 1, 0, 1 -> st", m.st(), "  grade -1", m.coeff(-1))

weights 0, 0, 0 -> <|5|₀>   plain mean 5.0
weights 1, 1, 1 -> <|5|₀>
weights 1, 0, 1 -> st 5.5   grade -1 -0.75


With weights `1, 0, 1` the value is the ordinary 5.5, and the zero weight is not
discarded: grade -1 is `-0.75`, which is `(4 - 5.5) / 2`, the amount the mean would
move per unit of weight if the zero-weighted item came back in. The rest of the tail is the
rest of that response: the full result is `5.5 - 0.75h + 0.375h**2 - ...`, which is
`(11 + 4w) / (2 + w)` expanded in the zero weight `w`, every order of it, so nothing
was linearised. Ordinary arithmetic cannot report any of it, because the item left
no trace.

## 6. A cancellation records what cancelled

Sections 3, 5 and this one are one fact: **a cancellation is a multiplication by
zero**. `6 - 6` is the same number as `6 * 0`, and dividing by zero gives the 6 back.
That is why `* 0` is reversible (section 3), and why two zeros keep their ratio
(section 5): `(a * 0) / (b * 0) = a / b`.

In [8]:
print("6 - 6 == 6 * 0       ", R(6) - R(6) == R(6) * 0, "   both", R(6) * 0)
print("(6 - 6) / 0 == 6     ", (R(6) - R(6)) / 0 == R(6))

6 - 6 == 6 * 0        True    both <|6|₋₁>
(6 - 6) / 0 == 6      True


Four things read as `0.0` in a float, and are four different numbers here:

In [9]:
for label, v in (("a written zero   R(0)", R(0)),
                 ("a cancellation   6 - 6", R(6) - R(6)),
                 ("another one      1 - 1", R(1) - R(1)),
                 ("a product        0 * 5", R(0) * R(5)),
                 ("nothing   Composite({})", Composite({}))):
    print("%-24s %-10s st() = %s" % (label, v, v.st()))

a written zero   R(0)    <|0|₀>     st() = 0.0
a cancellation   6 - 6   <|6|₋₁>    st() = 0.0
another one      1 - 1   <|1|₋₁>    st() = 0.0
a product        0 * 5   <|5|₋₁>    st() = 0.0
nothing   Composite({})  ∅          st() = 0.0


`6 - 6` is `|6|_-1`: six units of the infinitesimal, the quantity that was
annihilated, one grade down. `1 - 1` is `|1|_-1`, and `0 * 5` is `|5|_-1`: a zero that carries what it was
multiplied by. A written zero `R(0)` shows as
`|0|_0` and becomes one unit, `|1|_-1`, the moment it is used in an operation, which is
why `0 + 0` in section 7 is `|2|_-1`. `Composite({})` is NOTHING, the absence of a
value, which is not zero.
All of them have standard part 0, so code that only reads values sees no difference.

The difference shows when the zero is used again, as in section 5.

## 7. Absence is not zero: start accumulators from NOTHING

There is no additive identity. `5 + 0` is not `5`: the written zero is an
infinitesimal, and it stays. NOTHING is the identity, because adding nothing does
nothing. That has one everyday consequence: Python's `sum()` starts from the integer
`0`, which is a written zero.

In [10]:
print("5 + 0                 ", R(5) + 0)
print("5 + NOTHING           ", R(5) + Composite({}))
print("0 + 0                 ", R(0) + R(0))
print()
print("sum([R(1), R(2)])     ", sum([R(1), R(2)]), "  <- sum() started from 0")
print("from NOTHING          ", Composite({}) + R(1) + R(2))
print("sum(..., Composite({}))", sum([R(1), R(2)], Composite({})))

5 + 0                  <|5|₀ |1|₋₁>
5 + NOTHING            <|5|₀>
0 + 0                  <|2|₋₁>

sum([R(1), R(2)])      <|3|₀ |1|₋₁>   <- sum() started from 0
from NOTHING           <|3|₀>
sum(..., Composite({})) <|3|₀>


## 8. Ordering beyond the reals

Comparison is by dominance: the infinitesimal is smaller than every positive real,
its square is smaller still, and an infinity is larger than every real.

In [11]:
print("h < 1e-300      ", ZERO < R(1e-300))
print("h**2 < h        ", ZERO * ZERO < ZERO)
print("INF > 1e300     ", INF > R(1e300))
print()
print("sorted:", sorted([R(1), ZERO, INF, R(-2), ZERO * ZERO, R(0.5)]))

h < 1e-300       True
h**2 < h         True
INF > 1e300      True

sorted: [<|-2|₀>, <|1|₋₂>, <|1|₋₁>, <|0.5|₀>, <|1|₀>, <|1|₁>]


## 9. Switching it off: `conventional()`

Inside `conventional()` a zero that arrives as a value behaves classically: it is
absorbed by addition, a cancellation is an ordinary 0, and dividing by it raises.
Use it for code ported from somewhere that has an additive identity, or to compare
the two readings.

In [12]:
with conventional():
    print("5 + 0   ", R(5) + 0)
    print("6 - 6   ", R(6) - R(6))
    print("5 * 0   ", R(5) * 0)
    try:
        R(1) / R(0)
    except ZeroDivisionError:
        print("1 / 0    ZeroDivisionError")

5 + 0    <|5|₀>
6 - 6    <|0|₀>
5 * 0    <|0|₀>
1 / 0    ZeroDivisionError


Outside the block, every conversion of a zero is reported by a warning
(`CancellationWarning` and its relatives); this notebook silences them in the first
cell so the outputs stay readable.

## 10. What it costs

**No additive identity.** `5 + 0 != 5`, by design (section 7).

**Addition does not commute on a cancelling pair.** In `a - a` the annihilated
quantity is unambiguous; in `a + (-a)` it is read from the left operand, so the two
orders differ by a sign. Multiplication is untouched, and so is distributivity.

In [13]:
print("2 + (-2) =", R(2) + R(-2))
print("(-2) + 2 =", R(-2) + R(2))

2 + (-2) = <|2|₋₁>
(-2) + 2 = <|-2|₋₁>


**Addition does not associate when a partial sum is exactly zero.** In an
accumulator this is the one that matters. A partial sum that cancels to zero is an
event, and the number remembers it, so the result can depend on the order the data
arrived in. The value is the same either way; the tail is not. Here is the weighted
mean of section 5 with a negative weight:

In [14]:
print("(1 + -1) + 2 =", (R(1) + R(-1)) + R(2))
print("1 + (-1 + 2) =", R(1) + (R(-1) + R(2)))
print()
a = weighted_mean([2.0, 4.0, 9.0], [1.0, -1.0, 1.0])    # weights 1, -1: the denominator hits 0
b = weighted_mean([2.0, 9.0, 4.0], [1.0, 1.0, -1.0])    # same data, no partial sum is zero
print("weights 1, -1, 1         ->", str(a)[:60], "...")
print("same data, other order   ->", b)
print("same value:", a.st() == b.st())

(1 + -1) + 2 = <|2|₀ |1|₋₁>
1 + (-1 + 2) = <|2|₀>

weights 1, -1, 1         -> <|7|₀ |-7|₋₁ |7|₋₂ |-7|₋₃ |7|₋₄ |-7|₋₅ |7|₋₆ |-7|₋₇ |7|₋₈ |- ...
same data, other order   -> <|7|₀>
same value: True


If an accumulator must not depend on order, either keep partial sums away from
exact zero or read only `st()`; `conventional()` (section 9) also removes it.

**Equality is stricter.** `==` compares every grade, including zero terms that
record a cancellation, so two numbers with the same value can be unequal.

In [15]:
y = (R(1) + ZERO) - ZERO
print(y, "== R(1) ?", y == R(1), "   same value:", y.st() == R(1).st())

<|1|₀ |0|₋₁> == R(1) ? False    same value: True


**Speed.** Every operation is a small map from grades to coefficients. The comparison
below times the same operation on Python floats and on composites, with every operand
built before the clock starts, so only arithmetic is measured; building a composite costs
about as much again (20,000 `R(float(i))` take about 50 ms). Medians of five runs.

In [16]:
import statistics
N, RUNS = 20000, 5
fa, fb = [1.5] * N, [float(i) for i in range(N)]
ca, cb = [R(1.5) for _ in range(N)], [R(float(i)) for i in range(N)]    # built before timing

def median_ms(loop):
    times = []
    for _ in range(RUNS):
        t = time.perf_counter(); loop(); times.append(time.perf_counter() - t)
    return statistics.median(times) * 1e3

def mul(xs, ys):
    for x, y in zip(xs, ys): x * y
def add(xs, ys):
    for x, y in zip(xs, ys): x + y
def multiply_add(xs, ys, start):
    s = start
    for x, y in zip(xs, ys): s = s + x * y
    return s

print("%d operations each, dict backend, operands prebuilt, median of %d runs" % (N, RUNS))
for name, f, c in (("multiply", lambda: mul(fa, fb), lambda: mul(ca, cb)),
                   ("add", lambda: add(fa, fb), lambda: add(ca, cb)),
                   ("multiply-add", lambda: multiply_add(fa, fb, 0.0), lambda: multiply_add(ca, cb, Composite({})))):
    tf, tc = median_ms(f), median_ms(c)
    print("  %-13s float %5.2f ms   composite %6.1f ms   %4.0fx slower" % (name, tf, tc, tc / tf))
print("same value from the multiply-add:", multiply_add(ca, cb, Composite({})).st() == multiply_add(fa, fb, 0.0))

20000 operations each, dict backend, operands prebuilt, median of 5 runs


  multiply      float  0.38 ms   composite   49.7 ms    131x slower


  add           float  0.37 ms   composite   42.6 ms    115x slower


  multiply-add  float  0.47 ms   composite   99.3 ms    209x slower
same value from the multiply-add: True


## Summary

| | ordinary arithmetic | composite |
|---|---|---|
| ordinary numbers | the float result | the same float, at grade 0 |
| floating-point rounding | present | present, unchanged |
| `x * 0` | `0`, `x` lost | `x` one grade down; `/ 0` brings it back |
| `x / 0` | error | an infinity of definite order and size |
| `0 / 0` | indeterminate | the ratio of what the zeros came from |
| `a - a` | `0` | the annihilated quantity, one grade down |
| zero weights in an average | `nan` | the unweighted mean, plus each zero weight's pull |
| additive identity | `0` | NOTHING, `Composite({})`; `0` is an infinitesimal |
| `a + (-a)` vs `(-a) + a` | equal | differ by a sign |
| `(a + b) + c` vs `a + (b + c)` | equal | differ in the tail when a partial sum is exactly zero |
| speed | native | about 115x (add) and 130x (multiply) slower than the same operation on Python floats, about 200x for a multiply-add loop (dict backend, operands prebuilt; building a composite costs about as much again); the default sparse-dense backend is slower, notably for addition |
| classical behaviour | always | inside `conventional()` |